# 🎬 Netflix Content Strategy Analysis
**Role:** Data Analyst — Content Intelligence Team  
**Dataset:** Netflix Movies & TV Shows (Kaggle — shivamb)  
**Objective:** Understand content distribution, identify trends, and deliver 5 business recommendations to guide Netflix's content acquisition strategy.

---

## 📦 Step 0 — Install & Import Libraries

In [ ]:
# Install dataset directly from a public source
!pip install -q kaggle

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global plot style
plt.rcParams.update({
    'figure.facecolor': '#0a0a0a',
    'axes.facecolor':   '#1a1a1a',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#cccccc',
    'xtick.color':      '#cccccc',
    'ytick.color':      '#cccccc',
    'text.color':       '#ffffff',
    'grid.color':       '#2a2a2a',
    'grid.linewidth':   0.7,
    'font.size':        12,
    'axes.titlesize':   15,
    'axes.titleweight': 'bold',
})

RED    = '#E50914'
GOLD   = '#F5C518'
BLUE   = '#21A0DB'
GREEN  = '#57C785'
PURPLE = '#C77DFF'
ORANGE = '#F0853A'
PALETTE = [RED, GOLD, BLUE, GREEN, PURPLE, ORANGE, '#FF6B9D']

print('✅ Libraries loaded successfully')

## 📥 Load the Dataset
> Download from the public GitHub mirror of the Kaggle Netflix dataset.

In [ ]:
URL = 'https://raw.githubusercontent.com/allenkong221/netflix-titles-dataset/main/netflix_titles.csv'
df = pd.read_csv(URL)
print(f'✅ Dataset loaded — {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
## 🔍 Step 1 — Understand the Data
> As a new analyst, the first job is to understand what we're working with before touching anything.

In [ ]:
# --- 1.1 Shape & Column Overview ---
print('=' * 55)
print('  DATASET OVERVIEW')
print('=' * 55)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print('\nColumns & Data Types:')
print('-' * 40)
print(df.dtypes.to_string())

In [ ]:
# --- 1.2 Missing Values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing Count', ascending=False)

print('\nMissing Values:')
print(missing_df.to_string())

In [ ]:
# --- 1.3 Duplicate Check ---
dupes = df.duplicated().sum()
print(f'Duplicate rows: {dupes}')

# --- 1.4 Basic Stats ---
print('\nNumerical Column Summary:')
print(df[['release_year']].describe().round(1).to_string())

print('\nContent Type Split:')
print(df['type'].value_counts().to_string())

In [ ]:
# --- 1.5 Five-Line Data Summary ---
print('=' * 60)
print('  DATA SUMMARY (5 KEY FACTS)')
print('=' * 60)
total      = len(df)
movies     = (df['type'] == 'Movie').sum()
tvshows    = (df['type'] == 'TV Show').sum()
yr_min     = df['release_year'].min()
yr_max     = df['release_year'].max()
countries  = df['country'].dropna().apply(lambda x: x.split(',')[0].strip()).nunique()
dir_miss   = df['director'].isnull().mean() * 100

print(f'1. The dataset has {total:,} Netflix titles with 12 columns covering cast, country, rating, and more.')
print(f'2. Movies dominate at {movies:,} ({movies/total*100:.1f}%) vs {tvshows:,} TV Shows ({tvshows/total*100:.1f}%).')
print(f'3. Content spans {yr_min}–{yr_max}, but most titles were added to Netflix post-2015.')
print(f'4. Over {countries} unique countries are represented, led by the US, India, and UK.')
print(f'5. Director info is missing for {dir_miss:.1f}% of titles — common for TV shows with multiple directors.')

---
## 🧹 Step 2 — Clean the Data
> Document every decision. A good analyst never silently drops or fills data.

In [ ]:
# --- 2.1 Fill Missing Text Columns ---
# Decision: Fill with 'Unknown' rather than dropping rows.
# Dropping 'director' would remove ~31% of the dataset — too much data loss.
df['director'].fillna('Unknown', inplace=True)
df['cast'].fillna('Unknown', inplace=True)
df['country'].fillna('Unknown', inplace=True)

# Decision: Fill missing 'rating' with 'NR' (Not Rated) — a valid Netflix classification.
df['rating'].fillna('NR', inplace=True)

print('✅ Missing text columns filled.')

In [ ]:
# --- 2.2 Parse Dates ---
# Decision: Parse 'date_added' to datetime so we can extract year and month.
# errors='coerce' turns any unparseable entries into NaT (11 rows) instead of crashing.
df['date_added']  = pd.to_datetime(df['date_added'].str.strip(), format='%B %d, %Y', errors='coerce')
df['year_added']  = df['date_added'].dt.year
df['month_added'] = df['date_added'].dt.month

print('✅ date_added parsed. New columns: year_added, month_added')
print(df[['date_added','year_added','month_added']].dropna().head(3).to_string())

In [ ]:
# --- 2.3 Extract Primary Country ---
# Decision: Many titles have co-production countries (comma-separated).
# We take the first listed country as the 'primary origin' for grouping.
df['primary_country'] = df['country'].apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) and x != 'Unknown' else 'Unknown'
)

# --- 2.4 Extract Numeric Duration ---
# Decision: 'duration' stores '90 min' or '2 Seasons' as strings.
# Extract the integer so we can do numerical analysis.
df['duration_val'] = df['duration'].str.extract(r'(\d+)').astype(float)

# --- 2.5 Extract Primary Genre ---
# Decision: 'listed_in' is comma-separated. Take the first tag as primary genre.
df['primary_genre'] = df['listed_in'].apply(
    lambda x: x.split(',')[0].strip() if isinstance(x, str) else 'Unknown'
)

print('✅ Cleaning complete!')
print(f'Final shape: {df.shape}')
print(f'Remaining nulls: {df.isnull().sum().sum()}')

In [ ]:
# --- 2.6 Cleaning Decision Log ---
decisions = {
    'Column'  : ['director','cast','country','rating','date_added','country','duration','show_id'],
    'Action'  : ['Fill → Unknown','Fill → Unknown','Fill → Unknown','Fill → NR',
                 'Parse to datetime','Extract primary (first listed)',
                 'Extract numeric value','Retained as-is'],
    'Reason'  : [
        'Dropping 31% of rows would destroy too much data; Unknown is accurate',
        'Same logic; 9% missing — preserve rows',
        'Preserves 476 rows; Unknown excluded from country-specific analysis',
        'NR is a valid Netflix rating for unclassified content',
        'Enables year/month extraction; 11 bad rows become NaT (not dropped)',
        'Co-productions listed comma-separated; first = primary origin',
        '"90 min" → 90, "2 Seasons" → 2 for numerical analysis',
        'Unique ID — no analysis value but harmless to keep'
    ]
}
print(pd.DataFrame(decisions).to_string(index=False))

---
## 📊 Step 3 — Exploratory Data Analysis
> 5 business questions, answered with Pandas.

In [ ]:
# ── Q1: What are the most common genres on Netflix? ──────────────────────
print('Q1 — Top 10 Primary Genres on Netflix')
print('=' * 45)
genre_counts = df['primary_genre'].value_counts().head(10)
print(genre_counts.to_string())
print(f'\n→ Dramas alone account for {genre_counts["Dramas"]/len(df)*100:.1f}% of all content.')

In [ ]:
# ── Q2: Which rating category produces the longest average movie? ─────────
print('Q2 — Average Movie Duration by Content Rating (minutes)')
print('=' * 55)
avg_dur = (
    df[df['type'] == 'Movie']
    .groupby('rating')['duration_val']
    .agg(['mean', 'count'])
    .round(1)
    .sort_values('mean', ascending=False)
)
avg_dur.columns = ['Avg Duration (min)', 'Count']
print(avg_dur.to_string())
print('\n→ NC-17 films average 131.5 min — the longest of any rating category.')

In [ ]:
# ── Q3: How has Netflix's content library grown year-over-year? ───────────
print('Q3 — Titles Added to Netflix Per Year')
print('=' * 42)
yearly_growth = (
    df.groupby(['year_added', 'type'])
    .size()
    .unstack(fill_value=0)
    .astype(int)
)
yearly_growth = yearly_growth[yearly_growth.index.notna()].loc[2008:]
yearly_growth['Total'] = yearly_growth.sum(axis=1)
print(yearly_growth.to_string())
print('\n→ From 90 titles in 2015 to 2,349 in 2019 — a 26× increase in 4 years.')

In [ ]:
# ── Q4: How do the top 3 content-producing countries differ? ─────────────
print('Q4 — Movie vs TV Show Split: USA, India, UK')
print('=' * 50)
top3 = ['United States', 'India', 'United Kingdom']
country_split = (
    df[df['primary_country'].isin(top3)]
    .groupby(['primary_country', 'type'])
    .size()
    .unstack(fill_value=0)
)
country_split['Total'] = country_split.sum(axis=1)
country_split['Movie %'] = (country_split['Movie'] / country_split['Total'] * 100).round(1)
print(country_split.to_string())
print('\n→ India: 93% Movies (Bollywood). UK: 58% Movies (strong TV tradition).')

In [ ]:
# ── Q5: What does the typical Netflix TV show look like (seasons)? ────────
print('Q5 — TV Show Season Count Distribution')
print('=' * 42)
seasons = df[df['type'] == 'TV Show']['duration_val']
print(seasons.describe().round(2).to_string())
print('\nSeason breakdown:')
print(seasons.value_counts().sort_index().head(10).to_string())
print(f'\n→ {(seasons == 1).mean()*100:.1f}% of TV shows on Netflix have only 1 season.')

---
## 📈 Step 4 — Visualizations (7 Charts)
> Every chart has a title, axis labels, and a takeaway line.

In [ ]:
# ── Chart 1 — BAR: Movies vs TV Shows ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
fig.patch.set_facecolor('#0a0a0a')

counts = df['type'].value_counts()
bars = ax.bar(counts.index, counts.values, color=[RED, GOLD], width=0.45, zorder=3)

ax.set_title('Movies vs TV Shows on Netflix', pad=14)
ax.set_xlabel('Content Type')
ax.set_ylabel('Number of Titles')
ax.grid(axis='y', zorder=0)
ax.set_axisbelow(True)

for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 40,
            f'{int(bar.get_height()):,}', ha='center', fontsize=13, fontweight='bold', color='white')

pct = counts['Movie'] / counts.sum() * 100
ax.text(0.97, 0.95, f'Movies = {pct:.1f}% of library',
        transform=ax.transAxes, ha='right', va='top', fontsize=10, color=GOLD)

plt.tight_layout()
plt.show()
print('Insight: Netflix is primarily a movie platform, but TV is catching up.')

In [ ]:
# ── Chart 2 — LINE: Content added per year ────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0a0a')

yearly = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)
yearly = yearly[yearly.index.notna()].loc[2008:]

for col, color in zip(yearly.columns, [RED, GOLD]):
    ax.plot(yearly.index, yearly[col], color=color, lw=2.5,
            marker='o', ms=6, label=col, zorder=3)
    ax.fill_between(yearly.index, yearly[col], alpha=0.08, color=color)

ax.set_title('Netflix Titles Added Per Year (2008–2020)', pad=14)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Titles Added')
ax.legend(facecolor='#1a1a1a', edgecolor='#444', labelcolor='white', fontsize=11)
ax.grid(zorder=0)
ax.set_axisbelow(True)
ax.set_xticks(yearly.index)

plt.tight_layout()
plt.show()
print('Insight: Netflix entered a hyper-growth content phase between 2016–2019.')

In [ ]:
# ── Chart 3 — HISTOGRAM: Movie duration distribution ─────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0a0a0a')

movie_dur = df[df['type'] == 'Movie']['duration_val'].dropna()
ax.hist(movie_dur, bins=40, color=RED, edgecolor='#0a0a0a', linewidth=0.5, zorder=3, alpha=0.9)

ax.axvline(movie_dur.mean(),   color=GOLD,  lw=2, linestyle='--', label=f'Mean:   {movie_dur.mean():.0f} min')
ax.axvline(movie_dur.median(), color=GREEN, lw=2, linestyle=':',  label=f'Median: {movie_dur.median():.0f} min')

ax.set_title('Distribution of Movie Durations on Netflix', pad=14)
ax.set_xlabel('Duration (minutes)')
ax.set_ylabel('Number of Movies')
ax.legend(facecolor='#1a1a1a', edgecolor='#444', labelcolor='white', fontsize=11)
ax.grid(axis='y', zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()
print('Insight: The sweet spot for Netflix movies is 80–110 minutes. Very few exceed 150 min.')

In [ ]:
# ── Chart 4 — SCATTER: Release year vs year added ────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))
fig.patch.set_facecolor('#0a0a0a')

scatter_df = df[df['year_added'].notna()].copy()
colors_map = scatter_df['type'].map({'Movie': RED, 'TV Show': GOLD})
ax.scatter(scatter_df['release_year'], scatter_df['year_added'],
           c=colors_map, alpha=0.25, s=12, zorder=3)

ax.set_title('Original Release Year vs Year Added to Netflix', pad=14)
ax.set_xlabel('Original Release Year')
ax.set_ylabel('Year Added to Netflix')
ax.grid(zorder=0)
ax.set_axisbelow(True)

patches = [
    mpatches.Patch(color=RED,  label='Movie'),
    mpatches.Patch(color=GOLD, label='TV Show')
]
ax.legend(handles=patches, facecolor='#1a1a1a', edgecolor='#444', labelcolor='white', fontsize=11)

plt.tight_layout()
plt.show()
print('Insight: Netflix heavily favours recent content (post-2010). Very few pre-1990 titles.')

In [ ]:
# ── Chart 5 — PIE: Content rating breakdown ───────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))
fig.patch.set_facecolor('#0a0a0a')

rating_counts = df['rating'].value_counts().head(7)
wedges, texts, autotexts = ax.pie(
    rating_counts.values,
    labels=rating_counts.index,
    colors=PALETTE[:len(rating_counts)],
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.78,
    explode=[0.04] * len(rating_counts),
    textprops={'color': 'white', 'fontsize': 11},
    wedgeprops={'edgecolor': '#0a0a0a', 'linewidth': 1.5}
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_color('#0a0a0a')
    at.set_fontweight('bold')

ax.set_title('Netflix Content Rating Distribution', pad=16)

plt.tight_layout()
plt.show()
print('Insight: TV-MA + TV-14 = nearly 60% of all content. Netflix targets adult audiences.')

In [ ]:
# ── Chart 6 — HEATMAP: Monthly content additions ─────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor('#0a0a0a')

heat_df = df[df['year_added'] >= 2015].copy()
pivot = heat_df.groupby(['year_added', 'month_added']).size().unstack(fill_value=0)
pivot.index = pivot.index.astype(int)
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

sns.heatmap(
    pivot, ax=ax,
    cmap='YlOrRd',
    linewidths=0.4, linecolor='#080808',
    annot=True, fmt='d',
    annot_kws={'size': 9, 'color': '#0a0a0a', 'weight': 'bold'},
    cbar_kws={'label': 'Titles Added'}
)

ax.set_title('Monthly Content Addition Heatmap (2015–2020)', pad=14)
ax.set_xlabel('Month')
ax.set_ylabel('Year')
ax.tick_params(labelsize=11)

plt.tight_layout()
plt.show()
print('Insight: Q4 months (Oct–Dec) and Q1 (Jan) consistently see the highest content drops.')

In [ ]:
# ── Chart 7 — HORIZONTAL BAR: Top 10 countries ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor('#0a0a0a')

top_c = df[df['primary_country'] != 'Unknown']['primary_country'].value_counts().head(10)
bar_colors = [RED if i == 0 else '#555555' for i in range(len(top_c))]

bars = ax.barh(top_c.index[::-1], top_c.values[::-1], color=bar_colors[::-1], zorder=3)

ax.set_title('Top 10 Countries by Volume of Netflix Content', pad=14)
ax.set_xlabel('Number of Titles')
ax.set_ylabel('Country')
ax.grid(axis='x', zorder=0)
ax.set_axisbelow(True)

for i, (val, label) in enumerate(zip(top_c.values[::-1], top_c.index[::-1])):
    ax.text(val + 25, i, f'{val:,}', va='center', fontsize=10, color='white')

plt.tight_layout()
plt.show()
print('Insight: US leads with 2,032 titles. India is a distant but significant #2 at 777.')

---
## 📋 Step 5 — Business Insights Report
> *Memo to: Netflix Content Strategy Team*  
> *From: Data Analytics Division*  
> *Re: 5 Recommendations Based on Library Analysis*

---

In [ ]:
report = """
╔══════════════════════════════════════════════════════════════════════════╗
║         NETFLIX CONTENT STRATEGY — ANALYST RECOMMENDATIONS             ║
╠══════════════════════════════════════════════════════════════════════════╣

INSIGHT 1: Double Down on TV Originals
──────────────────────────────────────
Finding : Movies make up 68.4% of the library (Chart 1), but TV show
          additions are accelerating rapidly (Chart 2, 2016→2019).
Why it matters: TV shows generate higher engagement, longer viewing
          sessions, and reduce churn (subscribers wait for next seasons).
Recommendation: Shift content budget to 60/40 TV-to-Movie ratio in
          original productions, especially drama and thriller series.

──────────────────────────────────────────────────────────────────────────

INSIGHT 2: Time Content Drops Strategically (Q4 is King)
──────────────────────────────────────────────────────────
Finding : The monthly heatmap (Chart 6) shows October, November, and
          December consistently have the highest content additions across
          all years from 2015–2019.
Why it matters: Holiday seasons and year-end drive peak subscriber sign-ups.
          Releasing major titles in Q4 maximizes acquisition and retention.
Recommendation: Plan flagship original releases for Oct–Dec window.
          Use Jan as a secondary release window (post-holiday re-engagement).

──────────────────────────────────────────────────────────────────────────

INSIGHT 3: Invest in India — But Diversify Beyond Bollywood
────────────────────────────────────────────────────────────
Finding : India is the #2 content provider globally (Chart 7), but 93%
          of Indian content is movies (EDA Q4). Only 55 Indian TV shows
          are on the platform vs 753 movies.
Why it matters: Indian subscribers (and the diaspora globally) are a
          massive growth market. TV shows build stickier engagement.
Recommendation: Commission 20–30 Indian original TV series per year
          in regional languages (Tamil, Telugu, Malayalam) to tap
          audiences beyond Hindi-speaking markets.

──────────────────────────────────────────────────────────────────────────

INSIGHT 4: The Ideal Movie Length is 90–110 Minutes
────────────────────────────────────────────────────
Finding : The duration histogram (Chart 3) shows a tight cluster at
          80–120 min (mean ≈ 99 min, median ≈ 98 min). Very few titles
          fall outside this band on the platform.
Why it matters: Completion rates drop sharply for films over 130 min
          on streaming. Content that fits 90–110 min performs better
          in discovery algorithms.
Recommendation: Use 90–110 min as a soft editorial guideline for
          commissioned films. Flag acquisitions over 140 min for
          special curation (prestige/awards track).

──────────────────────────────────────────────────────────────────────────

INSIGHT 5: Expand Family & Kids Content to Defend Against Disney+
──────────────────────────────────────────────────────────────────
Finding : The ratings pie chart (Chart 5) shows TV-MA + TV-14 dominate
          at ~60% of all content. Family-safe content (TV-Y, TV-Y7, TV-G,
          PG) makes up less than 15% of the library.
Why it matters: Disney+, Apple TV+, and Peacock are aggressively targeting
          family audiences — a segment Netflix currently under-serves.
          Families are high-value, low-churn subscribers.
Recommendation: Increase family/kids content to at least 25% of annual
          new additions. Invest in animated originals and live-action
          family films rated TV-G through PG-13.

╚══════════════════════════════════════════════════════════════════════════╝
"""
print(report)

---
## ✅ Summary

| Step | What We Did |
|------|-------------|
| 1 — Load & Inspect | Loaded 6,234 rows × 12 cols, found missing values in 5 columns, zero duplicates |
| 2 — Clean | Filled NaNs, parsed dates, extracted country/genre/duration into new columns |
| 3 — EDA | Answered 5 business questions using groupby, value_counts, describe |
| 4 — Visualize | 7 charts: bar, line, histogram, scatter, pie, heatmap, horizontal bar |
| 5 — Insights | 5 data-backed recommendations for Netflix's content strategy |

> **Dataset:** Netflix Movies & TV Shows — [Kaggle (shivamb)](https://www.kaggle.com/datasets/shivamb/netflix-shows)